# 🧠 What I Want to Build
---

A chatbot that:

1. Takes a reference website (Liquid docs)
2. Converts it into a knowledge base
3. Answers questions only from that knowledge
4. If answer not found → responds:
    >“I don't have an answer for it”

## Improvement 1 - Cosine Similarity

**🔹 What it does**

Instead of distance, it measures angle between vectors:
```
cos(θ) = (A · B) / (|A| |B|)
```

**🧠 Intuition**
- Ignores magnitude
- Focuses on direction (meaning)

👉 Two sentences with same meaning → similar direction → high cosine similarity

## Improvement 2 — Semantic Chunking Idea

**🔹 What it does**

Merge sentences only if they are semantically similar

**🧠 How It Works**
- Split into sentences
- Convert each sentence → embedding
- Compare similarity between consecutive sentences
- If similarity is high → merge
- If low → start new chunk

## 🧱 Step-by-Step Implementation

#### 1. 📥 Ingest Liquid Documentation & Save Data

From this root:
👉 https://shopify.dev/docs/api/liquid

1. Visit the page
2. Extract all internal links
3. Visit each link
4. Avoid duplicates
5. Extract clean text
6. Store everything for chunking

In [8]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time

visited = set()
to_visit = set()

BASE_URL = "https://shopify.dev/docs/api/liquid"
DOMAIN = "shopify.dev"

to_visit.add(BASE_URL)

In [9]:
# Clean URLs
def clean_url(url):
    parsed = urlparse(url)
    return parsed.scheme + "://" + parsed.netloc + parsed.path

In [10]:
# Extract Only Useful Content
def extract_content(soup):
    main_content = soup.find("main")
    
    if main_content:
        text = main_content.get_text(separator=" ", strip=True)
    else:
        text = soup.get_text(separator=" ", strip=True)
    return text

In [11]:
# Crawl Loop
def crawl(max_pages=50):
    all_text = []

    while to_visit and len(visited) < max_pages:
        url = clean_url(to_visit.pop())

        if url in visited:
            continue

        print(f"Crawling: {url}")
        visited.add(url)

        try:
            response = requests.get(url, timeout=10)
            soup = BeautifulSoup(response.text, "html.parser")
            # Extract text
            text = extract_content(soup)
            # store data
            all_text.append({
             "url": url,
             "content": text
            })

            # Extract links
            for link in soup.find_all("a", href=True):
                href = link["href"]
                full_url = clean_url(urljoin(url, href))

                # Keep only internal links
                if DOMAIN in urlparse(full_url).netloc:
                    if full_url not in visited:
                        to_visit.add(full_url)

            time.sleep(1)  # polite crawling

        except Exception as e:
            print(f"Error: {e}")

    return all_text

In [12]:
# save data
def save_data(docs, path="data/liquid_docs-improved.json"):
    import os
    import json

    os.makedirs(os.path.dirname(path), exist_ok=True)

    with open(path, "w") as f:
        json.dump(docs, f)

    print(f"Saved {len(docs)} documents")


In [13]:
# Verify crawled data
docs = crawl(10)

print(f"\nTotal pages scraped: {len(docs)}")

# dump the data to avoid crawling everytime
save_data(docs)

Crawling: https://shopify.dev/docs/api/liquid
Crawling: https://shopify.dev/docs/api/liquid/tags/assign
Crawling: https://shopify.dev/themes/architecture/templates/product
Crawling: https://shopify.dev/docs/api/liquid/objects/articles
Crawling: https://shopify.dev/themes/tools/theme-check
Crawling: https://shopify.dev/docs/storefronts/themes/product-merchandising/recommendations
Crawling: https://shopify.dev/docs/storefronts/themes/tools/theme-check/commands
Crawling: https://shopify.dev/docs
Crawling: https://shopify.dev/docs/storefronts/themes/tools/theme-access
Crawling: https://shopify.dev/docs/themes/store/requirements

Total pages scraped: 10
Saved 10 documents


#### 2. Semantic based chunking

In [16]:
import re
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")


# 🔹 Step 1: Sentence split
def split_sentences(text):
    return re.split(r'(?<=[.!?]) +', text)


# 🔹 Step 2: Semantic chunking
def semantic_chunk(text, similarity_threshold=0.7, max_chunk_size=500):
    sentences = split_sentences(text)

    if not sentences:
        return []

    sentence_embeddings = model.encode(sentences)

    chunks = []
    current_chunk = sentences[0]

    for i in range(1, len(sentences)):
        prev_emb = sentence_embeddings[i - 1]
        curr_emb = sentence_embeddings[i]

        # cosine similarity
        similarity = np.dot(prev_emb, curr_emb) / (
            np.linalg.norm(prev_emb) * np.linalg.norm(curr_emb)
        )

        # decide whether to merge
        if similarity > similarity_threshold and len(current_chunk) < max_chunk_size:
            current_chunk += " " + sentences[i]
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentences[i]

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

#### 2: Embed + FAISS

In [17]:
import json
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load data
with open("data/liquid_docs-improved.json") as f:
    docs = json.load(f)

all_chunks = []
metadata = []

for doc in docs:
    chunks = semantic_chunk(doc["content"])
    for chunk in chunks:
        all_chunks.append(chunk)
        metadata.append(doc["url"])


# 🧠 Embedding
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(all_chunks)
# Normalize embeddings (for cosine similarity)
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)


# ⚡ FAISS (Inner Product = Cosine)
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(np.array(embeddings))


# 💾 Save everything
faiss.write_index(index, "data/index-improved.faiss")

with open("data/chunks-improved.json", "w") as f:
    json.dump({
        "chunks": all_chunks,
        "metadata": metadata
    }, f)

print("Embedding + Indexing complete")


Embedding + Indexing complete


#### 3: Chatbot (RAG + Guardrails)

In [ ]:
# chatbot.py

import json
import faiss
import numpy as np
import requests
from sentence_transformers import SentenceTransformer


# Load
index = faiss.read_index("data/index-improved.faiss")

with open("data/chunks-improved.json") as f:
    data = json.load(f)

chunks = data["chunks"]
metadata = data["metadata"]

model = SentenceTransformer("all-MiniLM-L6-v2")


# 🔍 Retrieval
def retrieve(query, k=3):
    q_vec = model.encode([query])
    distances, indices = index.search(np.array(q_vec), k)

    results = [chunks[i] for i in indices[0]]
    sources = [metadata[i] for i in indices[0]]
    scores = distances[0]

    return results, sources, scores


# 🚨 Relevance Check
THRESHOLD = 1.2  # tune this

def is_relevant(scores):
    return min(scores) < THRESHOLD


# 🤖 Prompt
def build_prompt(context, query):
    return f"""
You are a strict assistant.

Answer ONLY from the provided context.
If the answer is not present, say:
"I don't have an answer for it"

Context:
{context}

Question:
{query}

Answer:
"""


# 🧠 LLM
def ask_llm(prompt):
    res = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "mistral",
            "prompt": prompt,
            "stream": False
        }
    )
    return res.json()["response"]


# 🔁 Final Pipeline
def chatbot(query):
    results, sources, scores = retrieve(query)

    if not is_relevant(scores):
        return "I don't have an answer for it"

    context = "\n\n".join(results)
    prompt = build_prompt(context, query)

    answer = ask_llm(prompt)

    return f"{answer}\n\nSources:\n" + "\n".join(set(sources))


# 🧪 CLI Loop
if __name__ == "__main__":
    while True:
        q = input("\nAsk: ")
        print(chatbot(q))


Ask:  what is liquid


 Liquid is a template language created by Shopify.

Sources:
https://shopify.dev/docs/themes/store/requirements
https://shopify.dev/docs/api/liquid



Ask:  can you explain in detail about liquid


 Liquid is a template language that was created by Shopify to make it easier for developers to design dynamic web pages. It allows you to create and manipulate content, data, and variables within templates, which can then be output as HTML or XML.

Liquid includes various features such as loops, conditional statements, filters, and tags that help in creating complex and dynamic templates. Some examples of these features are:

* Loops: used to iterate over a list of items and output each item. Example: {% for product in products %} {{ product.title }} {% endfor %}
* Conditional statements: used to check if a condition is true or false, and execute code based on the result. Example: {% if product.available %} This product is available {% else %} This product is not available {% endif %}
* Filters: used to manipulate data before it is outputted. Examples include date formatting with `date: "%Y-%m-%d"` or truncating text with `truncatewords: 10`. Example: {{ product.description | truncatew


Ask:  where liquid is used?


...

 Liquid is a template language created by Shopify and it's primarily used for generating dynamic pages within the Shopify platform. It's not designed to be used directly with React, which is a JavaScript library for building user interfaces. If you want to use both, you would typically use Liquid to generate the basic structure of your HTML pages, and then use React to manage components and handle interactions within those pages. However, there are third-party solutions available that can help integrate Liquid and React more closely, such as GatsbyJS or Next.js with Shopify's APIs.

Sources:
https://shopify.dev/docs/api/liquid/tags/assign
https://shopify.dev/docs/themes/store/requirements
https://shopify.dev/docs/api/liquid



Ask:  what is react


 I don't have an answer for it in the provided context. "React" is a popular JavaScript library for building user interfaces, particularly for single-page applications. It's not directly related to Shopify or its theme functionality.

Sources:
https://shopify.dev/docs/themes/store/requirements
https://shopify.dev/docs/api/liquid



Ask:  tell me how can I add additional information for a variant when added to cart?


 To add additional information for a variant when added to the cart, you can use custom data attributes in your HTML markup for the quantity input and Add to cart button associated with each variant. When a customer adds the variant to their cart using the AJAX API, you can include this custom data as part of the request sent to the server. On the server side, you can then store and process this additional information accordingly.

For example:

```html
<div class="product-variant">
  <input type="hidden" name="custom_data[color]" value="red" />
  <input type="hidden" name="custom_data[size]" value="large" />
  <!-- Other variant details -->
  <input data-quantity-name="qty_color_red_size_large" class="add-to-cart-button" type="submit" value="Add to cart" />
</div>
```

In this example, the custom data attributes `custom_data[color]`, `custom_data[size]` store additional information about the variant (in this case, color and size), while the `data-quantity-name` attribute is used to sp